# Differential RV recovery (physical spectrum Doppler + fixed WAVE)

Minimal E2E check of the differential-RV requirement:

> If two datasets have injected RVs `(rv1, rv2)` with `drv = rv2 − rv1`, then after the DRP
> `|estimated_drv − drv| < 10 cm/s`.

Physical injection path (default `inject_rv_via='spectrum'`):

1. Densify the 1D BT-Settl template to R >= 3e5 (PCHIP, densify-only).
2. Doppler-shift that densified spectrum by `lambda * (1+z)`.
3. Paint onto the ZEMAX rate map.
4. Attach the **same** packaged line-ZP WAVE for every RV (no WAVE×(1+z)).

Sequence in this notebook:

1. **RV = 0 control** — recover ≈0 with Stage 0–2 (algorithmic zero point).
2. **Two injections** near −50 km/s separated by 5 m/s (same UTR seed).
3. Score `|estimated_drv − drv|` against the 10 cm/s requirement.

**Environment variables** (set in your shell / kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache |
| `KECK_OBSERVER_EMAIL` | Syncing from KOA | Keck Observer email |
| `KECK_OBSERVER_PASSWORD` | Syncing from KOA | Keck Observer password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/path/to/hispec_rv_recovery_sim"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Injection configuration

Fixed packaged WAVE for all runs. Spectrum Doppler paint only (`inject_rv_via='spectrum'`).


In [ ]:
FIBERS = [1]
INJECT_RV_VIA = "spectrum"  # densify → Doppler paint; WAVE fixed

# Absolute zero point (km/s) and differential (m/s)
RV_ZERO_POINT_KM_S = -50.0
DRV_M_S = 5.0  # rv2 − rv1

RV0_KM_S = 0.0
RV1_KM_S = RV_ZERO_POINT_KM_S
RV2_KM_S = RV_ZERO_POINT_KM_S + DRV_M_S / 1000.0

# Same seed so differential residual isolates algorithmic (not UTR noise) terms
SEED = 0

REQUIREMENT_M_S = 0.10  # 10 cm/s

print(f"fiber(s): {FIBERS}")
print(f"inject_rv_via = {INJECT_RV_VIA!r}  (fixed packaged WAVE)")
print(f"RV=0 control, seed={SEED}")
print(f"rv1 = {RV1_KM_S:.6f} km/s  ({RV1_KM_S * 1000.0:.3f} m/s)")
print(f"rv2 = {RV2_KM_S:.6f} km/s  ({RV2_KM_S * 1000.0:.3f} m/s)")
print(f"drv = rv2 − rv1 = {DRV_M_S:.3f} m/s")
print(f"requirement: |estimated_drv − drv| < {REQUIREMENT_M_S:.2f} m/s (10 cm/s)")


## 3. Helper: one injection → Stage 0 / 1 / 2

Stage-1 uses the in-memory packaged WAVE from `create_ramp_from_spectrum` (unscaled
when `inject_rv_via='spectrum'`). Traces / fiberflat come from the local caldb.


In [ ]:
from pathlib import Path

from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum


def run_one_injection(label: str, injected_rv_km_s: float, seed: int) -> dict:
    """Build one UTR, reduce Stage0–2, return recovered aggregate RV (m/s)."""
    run_dir = str(Path(OUT) / label)
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(f"\n=== {label}: inject {injected_rv_km_s:.6f} km/s, seed={seed} ===")
    ramp, cals, rate_map = create_ramp_from_spectrum(
        run_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename="lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5",
        mag_ab=3.05,
        filter_band="J",
        seed=seed,
        radial_velocity_km_s=injected_rv_km_s,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
    )
    wmeta = getattr(cals.get("wavesol"), "meta", None)
    print(
        f"ramp={ramp.meta.filename}  shape={ramp.data.shape}  "
        f"wavesol note={getattr(wmeta, 'note', None)}  "
        f"injected_rv_via={getattr(wmeta, 'injected_rv_via', None)}"
    )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=run_dir).apply(ramp)
    l1 = Stage1Pipeline(
        output_dir=run_dir,
        prim_args={
            "spec_extract": {"do_fibers": do_fibers},
            "set_wavesol": {"wavesol": cals["wavesol"]},
        },
    ).apply(l0)
    l2 = Stage2Pipeline(
        output_dir=run_dir,
        prim_args={
            "blaze_corr": {"do_fibers": do_fibers},
            "stellar_rv": {
                "ccf_input_velocity": injected_m_s,
                "do_fibers": list(FIBERS),
                "save_preview": False,
                "preview_write_png": False,
            },
            "sky_sub": {"run": False},
        },
    ).apply(l1)

    fib = FIBERS[0]
    avg = getattr(l2, f"rvccfavg_fib{fib}", None)
    if avg is None or len(avg) == 0:
        raise RuntimeError(f"{label}: missing rvccfavg_fib{fib}")
    rv_rec = float(avg["RV"][0])
    rv_err = float(avg["RV_ERR"][0])
    print(vpn.ucsd.edu
        f"fiber {fib}: recovered RV = {rv_rec:.9f} ± {rv_err:.9f} m/s  "
        f"(bias vs inject = {rv_rec - injected_m_s:.9f} m/s)"
    )
    return {
        "label": label,
        "run_dir": run_dir,
        "seed": seed,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "wavesol": cals["wavesol"],
        "l2": l2,
    }


## 4. RV = 0 control

Densify → paint at RV=0 → fixed packaged WAVE → Stage 0–2. Recovered aggregate RV
should be near zero (algorithmic zero point of the closed loop).


In [ ]:
result0 = run_one_injection("inj0_rv0", RV0_KM_S, SEED)
print(
    f"\nRV=0 control: recover = {result0['recover_m_s']:.12f} ± "
    f"{result0['recover_err_m_s']:.12f} m/s"
)


## 5. Two spectrum-Doppler injections (fixed WAVE)

Same densified-template Doppler paint path; WAVE is the packaged line-ZP bake for both.


In [ ]:
result1 = run_one_injection("inj1_rv1", RV1_KM_S, SEED)
result2 = run_one_injection("inj2_rv2", RV2_KM_S, SEED)

# Sanity: WAVE arrays should match across injections (fixed WAVE path)
import numpy as np

w1 = np.asarray(result1["wavesol"].wave_fib1, dtype=np.float64)
w2 = np.asarray(result2["wavesol"].wave_fib1, dtype=np.float64)
w0 = np.asarray(result0["wavesol"].wave_fib1, dtype=np.float64)
print(
    f"max |WAVE fib1 rv1−rv0| = {np.nanmax(np.abs(w1 - w0)):.3e}  "
    f"(should be ~0 for inject_rv_via='spectrum')"
)
print(
    f"max |WAVE fib1 rv2−rv1| = {np.nanmax(np.abs(w2 - w1)):.3e}  "
    f"(should be ~0 for inject_rv_via='spectrum')"
)


## 6. Differential RV score vs 10 cm/s requirement


In [ ]:
drv = result2["inject_m_s"] - result1["inject_m_s"]
estimated_drv = result2["recover_m_s"] - result1["recover_m_s"]
residual = estimated_drv - drv
passed = abs(residual) < REQUIREMENT_M_S

print("=== Absolute recovery ===")
print(
    f"rv0 inject = {result0['inject_m_s']:.9f} m/s  "
    f"recover = {result0['recover_m_s']:.9f} ± {result0['recover_err_m_s']:.9f} m/s  "
    f"bias = {result0['recover_m_s'] - result0['inject_m_s']:.9f} m/s"
)
print(
    f"rv1 inject = {result1['inject_m_s']:.9f} m/s  "
    f"recover = {result1['recover_m_s']:.9f} ± {result1['recover_err_m_s']:.9f} m/s  "
    f"bias = {result1['recover_m_s'] - result1['inject_m_s']:.9f} m/s"
)
print(
    f"rv2 inject = {result2['inject_m_s']:.9f} m/s  "
    f"recover = {result2['recover_m_s']:.9f} ± {result2['recover_err_m_s']:.9f} m/s  "
    f"bias = {result2['recover_m_s'] - result2['inject_m_s']:.9f} m/s"
)
print()
print("=== Differential RV recovery ===")
print(f"drv            = {drv:.12f} m/s")
print(f"estimated_drv  = {estimated_drv:.12f} m/s")
print(
    f"estimated_drv − drv = {residual:.12f} m/s  "
    f"({residual * 100.0:.9f} cm/s)"
)
print(f"requirement    = |estimated_drv − drv| < {REQUIREMENT_M_S:.2f} m/s (10 cm/s)")
print(f"PASS: {passed}" if passed else f"FAIL: {passed}")

# Report-only until spectrum-paint + fixed WAVE meets the differential budget.
# (WAVE×(1+z) previously met <10 cm/s; physical paint still shows large absolute bias.)
if not passed:
    print(
        f"WARNING: differential residual {residual:.12f} m/s exceeds "
        f"{REQUIREMENT_M_S:.2f} m/s — spectrum-paint transfer needs further diagnosis."
    )
